# Downstream consequences and novelty screening for certified mimetic operators

This notebook answers the two questions a referee is most likely to raise about the exact
operators. It makes no model calls, re-runs no search, and takes about a minute.

**Why does the trade-off matter?** The certified constructions are less accurate than the
MOLE/Castillo–Grone reference on the manufactured problem, so their value has to come from the
structure they recover. Sections B–D test that directly: whether the weighted energy is
guaranteed to decay, whether explicit time integration is affected, and whether a symmetric
system is available to a conjugate-gradient solver.

**Is the operator new?** Section A screens each certified operator against every family the
novelty library actually contains, up to scaling, sign and reflection, and refuses to license a
novelty claim while any required family is missing.

Everything is recomputed from the stored `operators/*.npz` arrays; nothing modifies the search
results.

> **Run order matters.** Section 1 is the *only* cell that defines `operators`. Run the cells
> top to bottom once. Re-running an earlier loader afterwards would silently replace the
> curated operator set with every archive found on disk, including the unpacked run caches,
> which duplicates the references and misattributes provenance.


## 1. Load the release package

The release keeps the four exact paper candidates in `derived/operators`, the follow-up
operators in `derived/followup_operators`, and the MOLE references only inside the run archives
under `runs/`. All three sources are gathered here and de-duplicated by program identity, so
the same operator stored under two naming conventions is counted once.

In [3]:
import json, math, re, shutil, zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUTPUT_DIR = Path("downstream_analysis")
NOVELTY_LIBRARY_DIRS = [Path("novelty_library"), Path("novelty_library_external")]
for folder in ("tables", "figures"): (OUTPUT_DIR / folder).mkdir(parents=True, exist_ok=True)


@dataclass
class Operator:
    name: str
    family: str            # 'reference' | 'candidate' | 'promoted' | 'followup'
    cells: int
    D: np.ndarray
    G: np.ndarray
    Q: np.ndarray | None
    P: np.ndarray | None
    metadata: dict = field(default_factory=dict)

    @property
    def L(self) -> np.ndarray:
        return self.D @ self.G

    def interior_laplacian(self) -> np.ndarray:
        """Dirichlet interior block: drop the two boundary scalar unknowns."""
        return self.L[1:-1, 1:-1]

    def interior_norm(self) -> np.ndarray | None:
        if self.Q is None: return None
        return self.Q[1:-1, 1:-1]


def _as_matrix(value) -> np.ndarray | None:
    if value is None: return None
    arr = np.asarray(value, dtype=float)
    return np.diag(arr) if arr.ndim == 1 else arr


def load_operators(root: Path) -> list[Operator]:
    """Every operator archive under one directory. Provenance is assigned by the caller."""
    operators: list[Operator] = []
    for path in sorted(Path(root).rglob("*_m*.npz")):
        with np.load(path, allow_pickle=True) as bundle:
            keys = set(bundle.files)
            if not {"D", "G"} <= keys: continue
            metadata = {}
            if "metadata_json" in keys:
                try: metadata = json.loads(str(bundle["metadata_json"]))
                except Exception: metadata = {}
            operators.append(Operator(
                name=path.stem, family="candidate", cells=int(bundle["cells"]),
                D=np.asarray(bundle["D"], dtype=float), G=np.asarray(bundle["G"], dtype=float),
                Q=_as_matrix(bundle["Q"] if "Q" in keys else (bundle["q"] if "q" in keys else None)),
                P=_as_matrix(bundle["P"] if "P" in keys else (bundle["p"] if "p" in keys else None)),
                metadata=metadata))
    return operators


PACKAGE_ROOT = Path("Verifier_Guided_Mimetic_Operators_Reproducibility_Package_Final_Clean")
if not PACKAGE_ROOT.exists():
    bundles = sorted(Path(".").glob("*Reproducibility_Package_Final_Clean*.zip"))
    if not bundles:
        raise FileNotFoundError(
            "Put the release zip beside this notebook, or set PACKAGE_ROOT to the unpacked folder."
        )
    with zipfile.ZipFile(bundles[-1]) as bundle: bundle.extractall(".")
    PACKAGE_ROOT = next(p for p in Path(".").glob("*Reproducibility_Package_Final_Clean*")
                        if p.is_dir())

# The MOLE references live only inside the run archives; unpack them once.
RUN_CACHE = PACKAGE_ROOT / "runs" / "_unpacked"
for archive in sorted((PACKAGE_ROOT / "runs").glob("*.zip")):
    target = RUN_CACHE / archive.stem
    if not target.exists():
        with zipfile.ZipFile(archive) as bundle: bundle.extractall(target)

SOURCE_ROOTS = [
    (PACKAGE_ROOT / "derived" / "operators", "promoted"),
    (PACKAGE_ROOT / "derived" / "followup_operators", "followup"),
    (RUN_CACHE / "v11_original_results", None),   # references and closed-grid candidates
]


def classify(stem: str, default: str | None) -> str:
    if "reference_mole" in stem: return "reference"
    return default or "candidate"


def program_key(operator: Operator) -> str:
    """Identify the same operator across naming conventions.

    The release stores a paper candidate as `<condition>_prog_...` while the run archive
    stores the same arrays as `search_prog_...`. Splitting the stem on "_m" is unsafe:
    several names contain "_min_" or "_moment_", and "reference_mole_k6" contains "_m"
    inside "mole", which would collapse all four references onto a single key.
    """
    identifier = operator.metadata.get("program_id")
    if identifier: return f"{identifier}@{operator.cells}"
    stem = re.sub(r"_m\d+$", "", operator.name)
    tail = re.search(r"[0-9a-f]{8,}$", stem)
    return f"{tail.group(0) if tail else stem}@{operator.cells}"


operators, seen_keys = [], set()
for root, default_family in SOURCE_ROOTS:
    if not root.exists(): continue
    for candidate in load_operators(root):
        candidate.family = classify(candidate.name, default_family)
        key = program_key(candidate)
        # Release-first ordering: those copies carry the archive condition and the
        # exact-certificate metadata, so the first hit wins.
        if key in seen_keys: continue
        seen_keys.add(key)
        operators.append(candidate)

RESULTS_ROOT = PACKAGE_ROOT
inventory = pd.DataFrame([{
    "operator": op.name, "provenance": op.family, "cells": op.cells,
    "archive_condition": op.metadata.get("archive_condition"),
    "norm_class": op.metadata.get("norm_class", "diagonal"),
    "target_order": op.metadata.get("target_order"),
    "boundary_order": op.metadata.get("boundary_order") or op.metadata.get("left_boundary_order"),
    "model": op.metadata.get("model_id"),
    "has_norm": op.Q is not None,
} for op in operators]).sort_values(["provenance", "operator"]).reset_index(drop=True)

EXPECTED = {"promoted": 4, "reference": 4, "followup": 3}
counts = inventory.provenance.value_counts().to_dict()
print(f"{len(operators)} operators from {PACKAGE_ROOT.name}")
print("provenance:", counts)
for family, expected in EXPECTED.items():
    if counts.get(family, 0) != expected:
        print(f"  WARNING: expected {expected} {family} operators, found {counts.get(family, 0)}. "
              "If this cell was run after another loader, restart the kernel and run once.")
display(inventory)

18 operators from Verifier_Guided_Mimetic_Operators_Reproducibility_Package_Final_Clean
provenance: {'candidate': 7, 'promoted': 4, 'reference': 4, 'followup': 3}


,operator,provenance,cells,archive_condition,norm_class,target_order,boundary_order,model,has_norm
0,candidate_k2_b1_r1_s3_m200,candidate,200,None,diagonal,NaN,1,None,True
1,candidate_k4_b2_r4_s7_m200,candidate,200,None,diagonal,NaN,2,None,True
2,candidate_k6_b3_r8_s11_m200,candidate,200,None,diagonal,NaN,3,None,True
3,candidate_k8_b4_r9_s13_m200,candidate,200,None,diagonal,NaN,4,None,True
4,regression_prog_k6_block_psd_reflection_hybrid...,candidate,200,None,block_psd,6.0,3,None,True
5,regression_prog_k8_block_psd_asymmetric_hybrid...,candidate,200,None,block_psd,8.0,4,None,True
6,search_prog_k6_block_psd_reflection_hybrid_0cc...,candidate,200,None,block_psd,6.0,4,None,True
7,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,200,None,block_psd,6.0,3,None,True
8,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,200,None,block_psd,6.0,4,None,True
9,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,200,None,diagonal,6.0,3,None,True


## 2. Novelty screening

Two comparisons run side by side. The **block comparison** uses the mesh-independent corner
blocks of the divergence and gradient, normalised for scale and sign and tested against
reflection, which is the equivalence a reader means by "the same operator"; shapes must agree
for a match, since differing row counts or support widths mean different constructions. The
**alignment-free fingerprint** compares invariants that survive any relabelling — leading scaled
eigenvalues and corner singular values — and catches near-matches a direct block comparison
would miss.

The gate is the point. `novelty_claim_supported` is true only when the library contains every
required family *and* nothing matched. While families are missing, the only defensible statement
is that the operator differs from what was actually loaded.

In [4]:
def boundary_blocks(operator: Operator, rows: int | None = None,
                    support: int | None = None) -> dict[str, np.ndarray]:
    """Mesh-independent corner blocks of D and G.

    The interior of a staggered operator is a translation-invariant stencil, so the
    only structure that distinguishes two constructions of the same interior order is
    the boundary closure. Comparing corners rather than whole matrices is what makes a
    comparison across published families possible at all.
    """
    meta = operator.metadata

    def as_int_list(value, fallback: list[int]) -> list[int]:
        """Metadata is not type-stable across archives.

        The release stores a support profile as the string "(10, 10, 10, 10, 10, 10)"
        while the run archives store a tuple, and taking max() of the string returns a
        character. In v2 that silently produced 7x3 corner windows for two of the four
        paper candidates, so their alignment-free fingerprints had too few singular
        values and every comparison came back infinite.
        """
        if value is None: return fallback
        if isinstance(value, str):
            found = [int(x) for x in re.findall(r"\d+", value)]
            return found or fallback
        if isinstance(value, (int, float)): return [int(value)]
        try: return [int(x) for x in value] or fallback
        except Exception: return fallback

    rows = rows or as_int_list(meta.get("left_rows") or meta.get("boundary_rows"), [6])[0]
    support = support or max(as_int_list(
        meta.get("left_support_profile") or meta.get("support_width"), [12]))
    rows = max(1, min(int(rows) + 1, operator.D.shape[0] // 3))
    support = max(2, min(int(support) + 2, operator.D.shape[1] // 3))
    return {"D_corner": operator.D[:rows, :support].copy(),
            "G_corner": operator.G[:rows, :support].copy()}


def canonical_form(block: np.ndarray) -> np.ndarray:
    """Scale- and sign-normalised block, for comparison up to those equivalences."""
    flat = block.astype(float)
    scale = np.max(np.abs(flat))
    if scale == 0: return flat
    normalised = flat / scale
    # Fix the sign by the first entry that is clearly non-zero.
    for value in normalised.ravel():
        if abs(value) > 1e-12:
            if value < 0: normalised = -normalised
            break
    return normalised


def block_distance(a: np.ndarray, b: np.ndarray) -> tuple[float, bool]:
    """Distance up to scale, sign and reflection, on the common corner window.

    Returning ``inf`` whenever the closures use different row counts or support widths
    would make every comparison vacuous, since that is exactly what differs between
    constructions. Instead the blocks are compared on the largest window both share,
    and the shape agreement is reported separately: two operators are the same only if
    the shapes agree and the distance is small.
    """
    rows = min(a.shape[0], b.shape[0]); cols = min(a.shape[1], b.shape[1])
    if rows == 0 or cols == 0: return float("inf"), False
    same_shape = a.shape == b.shape
    ca = canonical_form(a[:rows, :cols])
    view = b[:rows, :cols]
    candidates = [canonical_form(view), canonical_form(view[::-1, ::-1]),
                  canonical_form(-view), canonical_form(-view[::-1, ::-1])]
    return float(min(np.max(np.abs(ca - c)) for c in candidates)), same_shape


FINGERPRINT_LENGTH = 6


def _padded_singular_values(block: np.ndarray, length: int = FINGERPRINT_LENGTH) -> list[float]:
    values = np.linalg.svd(block, compute_uv=False)
    values = values / max(float(np.max(np.abs(block))), 1e-300)
    padded = np.zeros(length)
    padded[:min(length, values.size)] = values[:length]
    return np.round(padded, 9).tolist()


def spectral_fingerprint(operator: Operator, cells: int | None = None) -> dict[str, Any]:
    """Alignment-free invariants: what survives any relabelling of the construction."""
    interior = operator.interior_laplacian()
    h = 1.0 / operator.cells
    eigenvalues = np.linalg.eigvals(interior)
    order = np.argsort(-eigenvalues.real)
    eigenvalues = eigenvalues[order]
    blocks = boundary_blocks(operator)
    return {
        "scaled_spectral_radius": float(np.max(np.abs(eigenvalues)) * h * h),
        "nonreal_modes": int(np.sum(np.abs(eigenvalues.imag) > 1e-8 * np.max(np.abs(eigenvalues)))),
        "leading_eigenvalues": np.round(np.sort(eigenvalues.real)[-6:] * h * h, 9).tolist(),
        # Padded to a fixed length so that two operators with different closure widths
        # remain comparable; a shorter spectrum is padded with zeros rather than making
        # the comparison undefined.
        "D_corner_singular_values": _padded_singular_values(blocks["D_corner"]),
        "G_corner_singular_values": _padded_singular_values(blocks["G_corner"]),
    }


def fingerprint_distance(a: dict[str, Any], b: dict[str, Any]) -> float:
    """Relative distance between two alignment-free fingerprints."""
    keys = ("leading_eigenvalues", "D_corner_singular_values", "G_corner_singular_values")
    worst = 0.0
    for key in keys:
        u, v = np.asarray(a[key], float), np.asarray(b[key], float)
        if u.shape != v.shape: return float("inf")
        denominator = np.maximum(np.abs(u), 1e-12)
        worst = max(worst, float(np.max(np.abs(u - v) / denominator)))
    return worst


def load_novelty_library(paths: list[Path]) -> list[dict[str, Any]]:
    """External families supplied as JSON; see the schema note in the notebook."""
    library = []
    for path in paths:
        if not path.exists(): continue
        for entry_path in sorted(path.glob("*.json")):
            try: entry = json.loads(entry_path.read_text())
            except Exception: continue
            if "D_corner" in entry and "G_corner" in entry:
                entry["D_corner"] = np.asarray(entry["D_corner"], float)
                entry["G_corner"] = np.asarray(entry["G_corner"], float)
                entry.setdefault("family", entry_path.stem)
                entry["source_file"] = str(entry_path)
                library.append(entry)
    return library


REQUIRED_FAMILIES = ("castillo_grone", "corbino_castillo", "strand",
                     "mattsson_nordstrom", "generalized_sbp_block_norm")


def load_family_declarations(paths: list[Path]) -> dict[str, dict[str, Any]]:
    """Explicit statements that a family cannot be compared, with a reason.

    Several published families live on collocated grids, where a staggered mimetic
    closure has no faithful coefficient-level image. Leaving those permanently
    "missing" would mean the gate can never close even after the comparison has been
    done as carefully as it can be. A declaration file records the judgement instead:
    {"family": "strand", "applicable": false, "justification": "..."}.
    """
    declarations: dict[str, dict[str, Any]] = {}
    for path in paths:
        if not path.exists(): continue
        for entry_path in sorted(path.glob("*.json")):
            try: entry = json.loads(entry_path.read_text())
            except Exception: continue
            if "applicable" in entry and "family" in entry:
                entry["source_file"] = str(entry_path)
                declarations[str(entry["family"]).lower()] = entry
    return declarations


def novelty_report(operators: list[Operator], library: list[dict[str, Any]],
                   declarations: dict[str, dict[str, Any]] | None = None,
                   match_tolerance: float = 1e-6) -> dict[str, Any]:
    """Screen certified operators against everything available, and gate the claim.

    A negative result is not evidence of novelty unless the library contains the
    families a reader would ask about. Each required family is therefore resolved to
    one of three states: compared against loaded coefficients, declared
    not-applicable with a written justification, or still outstanding.
    """
    declarations = declarations or {}
    loaded = {str(entry.get("family", "")).lower() for entry in library}
    status = {}
    for family in REQUIRED_FAMILIES:
        if any(family in name for name in loaded): status[family] = "compared"
        elif family in declarations and not declarations[family].get("applicable", True):
            status[family] = "declared_not_applicable"
        else: status[family] = "outstanding"
    outstanding = [f for f, state in status.items() if state == "outstanding"]
    rows = []
    for operator in operators:
        blocks = boundary_blocks(operator)
        for entry in library:
            d_distance, d_shape = block_distance(blocks["D_corner"], entry["D_corner"])
            g_distance, g_shape = block_distance(blocks["G_corner"], entry["G_corner"])
            distance = max(d_distance, g_distance)
            same_shape = bool(d_shape and g_shape)
            rows.append({"operator": operator.name, "library_family": entry.get("family"),
                         "block_distance": distance, "same_block_shape": same_shape,
                         "match": bool(same_shape and distance < match_tolerance)})
    return {
        "comparisons": rows,
        "family_status": status,
        "library_families_loaded": sorted(loaded),
        "required_families_outstanding": outstanding,
        "declared_not_applicable": {f: declarations[f].get("justification", "")
                                    for f, state in status.items()
                                    if state == "declared_not_applicable"},
        "library_complete": not outstanding,
        "any_match": any(row["match"] for row in rows),
        "novelty_claim_supported": bool(rows) and not outstanding and not any(r["match"] for r in rows),
        "note": ("A novelty claim requires every required family to be either compared "
                 "against loaded coefficients or declared not applicable with a written "
                 "justification, and no match. While families are outstanding, the only "
                 "defensible statement is that the operator differs from the families "
                 "actually loaded."),
    }

In [5]:
COEFFICIENT_ENTRY_TEMPLATE = {
    "family": "corbino_castillo_k6",
    "reference": "Corbino & Castillo (2020), JCAM 364:112326, Table 2",
    "target_order": 6,
    "topology": "staggered",
    "D_corner": [[0.0, 0.0]],
    "G_corner": [[0.0, 0.0]],
}
DECLARATION_TEMPLATE = {
    "family": "strand",
    "applicable": False,
    "reference": "Strand (1994), JCP 110:47-67",
    "justification": ("Collocated-grid SBP operators: the first-derivative closure has no "
                      "faithful coefficient-level image on the staggered scalar/vector pair "
                      "used here, so a block-level comparison is not defined. Recorded as "
                      "not comparable rather than left outstanding."),
}
LIBRARY_DIR = Path("novelty_library"); LIBRARY_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "novelty_library_coefficient_template.json").write_text(
    json.dumps(COEFFICIENT_ENTRY_TEMPLATE, indent=2))
(OUTPUT_DIR / "novelty_library_declaration_template.json").write_text(
    json.dumps(DECLARATION_TEMPLATE, indent=2))

LIBRARY_SEARCH_PATHS = [PACKAGE_ROOT / "derived" / "novelty_library",
                        RUN_CACHE / "v11_original_results" / "open_program" /
                        "novelty_library_external",
                        *NOVELTY_LIBRARY_DIRS]
library = load_novelty_library(LIBRARY_SEARCH_PATHS)
declarations = load_family_declarations(LIBRARY_SEARCH_PATHS)

# The reproduced reference operators are a family we already hold, so they enter the
# library automatically; no published table is needed for them.
for op in operators:
    if op.family != "reference": continue
    blocks = boundary_blocks(op, rows=8, support=12)
    library.append({"family": f"castillo_grone_{op.name.split('_')[-2]}",
                    "reference": "reproduced from MOLE in this run",
                    "D_corner": blocks["D_corner"], "G_corner": blocks["G_corner"]})

screened = [op for op in operators if op.family in ("promoted", "followup")]
report = novelty_report(screened, library, declarations)
comparisons = pd.DataFrame(report["comparisons"])
if not comparisons.empty:
    comparisons.to_csv(OUTPUT_DIR / "tables" / "novelty_block_comparisons.csv", index=False)
    display(comparisons.sort_values("block_distance").groupby("operator").head(1)
            .reset_index(drop=True))

fingerprints = {op.name: spectral_fingerprint(op) for op in operators}
pairs = [{"operator": op.name, "compared_with": other.name, "provenance": other.family,
          "fingerprint_distance": fingerprint_distance(fingerprints[op.name],
                                                       fingerprints[other.name])}
         for op in screened for other in operators if other.name != op.name]
fingerprint_table = pd.DataFrame(pairs).sort_values("fingerprint_distance")
fingerprint_table.to_csv(OUTPUT_DIR / "tables" / "novelty_fingerprint_distances.csv", index=False)
undefined = int(np.isinf(fingerprint_table.fingerprint_distance).sum())
if undefined:
    print(f"WARNING: {undefined} fingerprint comparisons are undefined; the alignment-free "
          "screen does not cover those operators.")
display(fingerprint_table.groupby("operator").head(1).reset_index(drop=True))

novelty_gate = {k: v for k, v in report.items() if k != "comparisons"}
(OUTPUT_DIR / "tables" / "novelty_gate.json").write_text(json.dumps(novelty_gate, indent=2))
print(json.dumps(novelty_gate, indent=2))
if report["required_families_outstanding"]:
    print("\nNOVELTY CLAIM BLOCKED. Outstanding families:",
          report["required_families_outstanding"])
    print(f"Add one file per family to {LIBRARY_DIR}/ : either coefficients, using the "
          "template at", OUTPUT_DIR / "novelty_library_coefficient_template.json",
          "or a not-applicable declaration, using",
          OUTPUT_DIR / "novelty_library_declaration_template.json")

,operator,library_family,block_distance,same_block_shape,match
0,v12_1_random_prog_k6_block_psd_reflection_min_...,castillo_grone_k4,0.202581,False,False
1,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,castillo_grone_k4,0.211844,False,False
2,structure_only_prog_k6_block_psd_reflection_mi...,castillo_grone_k4,0.344390,False,False
3,full_metrics_prog_k6_block_psd_reflection_hybr...,castillo_grone_k4,0.405780,False,False
4,no_archive_prog_k6_block_psd_asymmetric_hybrid...,castillo_grone_k4,0.497318,False,False
5,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,castillo_grone_k2,0.560355,False,False
6,illumination_prog_k6_block_psd_reflection_hybr...,castillo_grone_k2,1.000000,False,False


,operator,compared_with,provenance,fingerprint_distance
0,full_metrics_prog_k6_block_psd_reflection_hybr...,search_prog_k6_block_psd_reflection_hybrid_0cc...,candidate,0.043479
1,structure_only_prog_k6_block_psd_reflection_mi...,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,0.084129
2,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,search_prog_k6_block_psd_reflection_hybrid_0cc...,candidate,0.285468
3,no_archive_prog_k6_block_psd_asymmetric_hybrid...,reference_mole_k4_m200,reference,0.344847
4,illumination_prog_k6_block_psd_reflection_hybr...,regression_prog_k6_block_psd_reflection_hybrid...,candidate,0.370668
5,v12_1_random_prog_k6_block_psd_reflection_min_...,regression_prog_k6_block_psd_reflection_hybrid...,candidate,0.526617
6,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,candidate_k4_b2_r4_s7_m200,candidate,0.871945


{
  "family_status": {
    "castillo_grone": "compared",
    "corbino_castillo": "outstanding",
    "strand": "outstanding",
    "mattsson_nordstrom": "outstanding",
    "generalized_sbp_block_norm": "outstanding"
  },
  "library_families_loaded": [
    "castillo_grone_k2",
    "castillo_grone_k4",
    "castillo_grone_k6",
    "castillo_grone_k8"
  ],
  "required_families_outstanding": [
    "corbino_castillo",
    "strand",
    "mattsson_nordstrom",
    "generalized_sbp_block_norm"
  ],
  "declared_not_applicable": {},
  "library_complete": false,
  "any_match": false,
  "novelty_claim_supported": false,
  "note": "A novelty claim requires every required family to be either compared against loaded coefficients or declared not applicable with a written justification, and no match. While families are outstanding, the only defensible statement is that the operator differs from the families actually loaded."
}

NOVELTY CLAIM BLOCKED. Outstanding families: ['corbino_castillo', 'strand', 'm

## 3. What the certified norm buys: guaranteed energy decay

For the semi-discrete heat equation $\dot u = L_I u$, the weighted energy satisfies

$$\frac{d}{dt}\,u^{\top} Q_I u = u^{\top}\!\left(Q_I L_I + L_I^{\top} Q_I\right) u,$$

so the largest eigenvalue of the symmetric part of $Q_I L_I$ is the guaranteed growth rate over
all initial data, and its eigenvector is the worst case. A valid weighted extended-Gauss
identity makes this non-positive; the fixed-closure obstruction says it cannot be non-positive
for the order-six and order-eight reference closures. No time stepping is needed for the bound.

Random initial data cannot discriminate — everything diffuses — so the simulation starts from
the worst-case datum. Note that the reference uses the diagonal weights stored with it, which
is the norm the low-order theory intends; ruling out *all* norms is the proposition's job, not
this experiment's.

In [6]:
RK_STABILITY = {
    "euler": lambda z: 1 + z,
    "rk2": lambda z: 1 + z + z ** 2 / 2,
    "rk4": lambda z: 1 + z + z ** 2 / 2 + z ** 3 / 6 + z ** 4 / 24,
}


def max_stable_step(eigenvalues: np.ndarray, scheme: str = "rk4",
                    tolerance: float = 1e-10) -> tuple[float, complex]:
    """Largest dt with every dt*lambda inside the absolute-stability region.

    Bisection on dt using the scheme's stability polynomial. The binding eigenvalue is
    returned as well: for these operators it answers whether the non-real boundary
    modes, rather than the resolved interior modes, set the time-step restriction.
    """
    amplification = RK_STABILITY[scheme]
    def stable(dt: float) -> bool:
        return bool(np.all(np.abs(amplification(dt * eigenvalues)) <= 1 + 1e-12))
    low, high = 0.0, 1.0
    while stable(high) and high < 1e6: high *= 2
    while high - low > tolerance * max(1.0, high):
        middle = 0.5 * (low + high)
        if stable(middle): low = middle
        else: high = middle
    binding = eigenvalues[int(np.argmax(np.abs(amplification(low * eigenvalues))))]
    return low, complex(binding)

In [7]:
def energy_growth_bound(operator: Operator) -> dict[str, Any]:
    """Sharp statement about the weighted energy, with no time stepping involved.

    The Q-weighted energy obeys d/dt (u^T Q u) = u^T (QL + L^T Q) u, so the largest
    eigenvalue of the symmetric part of Q_I L_I is the guaranteed growth rate over all
    initial data. It is non-positive exactly when a valid weighted extended-Gauss
    identity holds, which is what the fixed-closure obstruction denies at orders six
    and eight. The corresponding eigenvector is the worst-case initial datum.
    """
    interior = operator.interior_laplacian()
    weight = operator.interior_norm()
    if weight is None: return {"operator": operator.name, "energy_bound_available": False}
    weight = 0.5 * (weight + weight.T)
    weight_spectrum = np.linalg.eigvalsh(weight)
    product = weight @ interior
    symmetric_part = 0.5 * (product + product.T)
    values, vectors = np.linalg.eigh(symmetric_part)
    scale = float(np.max(np.abs(values)))
    return {
        "operator": operator.name, "family": operator.family, "energy_bound_available": True,
        "weight_positive_definite": bool(weight_spectrum[0] > 0),
        "weight_min_eigenvalue": float(weight_spectrum[0]),
        "max_energy_growth_rate": float(values[-1]),
        "scaled_growth_rate": float(values[-1] / scale) if scale else float("nan"),
        "energy_decay_guaranteed": bool(values[-1] <= 0 and weight_spectrum[0] > 0),
        "worst_case_vector": vectors[:, -1],
    }


def energy_history(operator: Operator, steps: int = 4000, dt_fraction: float = 0.4,
                   scheme: str = "rk4", seed: int = 0,
                   initial: str = "worst_case") -> dict[str, Any]:
    """Track the Q-weighted energy of the semi-discrete heat equation.

    For an operator admitting a positive weighted extended-Gauss identity the energy
    is non-increasing. The reference closures at orders six and eight do not admit
    one, so this is the observable that the obstruction predicts.
    """
    interior = operator.interior_laplacian()
    weight = operator.interior_norm()
    if weight is None: return {"operator": operator.name, "energy_available": False}
    weight = 0.5 * (weight + weight.T)
    eigenvalues = np.linalg.eigvals(interior)
    dt, _ = max_stable_step(eigenvalues, scheme)
    dt *= dt_fraction
    amplification = RK_STABILITY[scheme]
    # Random data diffuses whatever the operator does, so it cannot discriminate.
    # The worst-case datum is the eigenvector realising the growth bound above.
    bound = energy_growth_bound(operator)
    if initial == "worst_case" and bound.get("energy_bound_available"):
        state = np.asarray(bound["worst_case_vector"], float).copy()
    else:
        state = np.random.default_rng(seed).standard_normal(interior.shape[0])
    state /= np.linalg.norm(state)
    identity = np.eye(interior.shape[0])
    # For a linear autonomous system the Runge-Kutta update is exactly the stability
    # polynomial applied to the operator: u^{n+1} = R(dt L) u^n.
    degree = {"rk4": 4, "rk2": 2, "euler": 1}[scheme]
    step_operator = sum(np.linalg.matrix_power(dt * interior, power) / math.factorial(power)
                        for power in range(degree + 1))
    energies = []
    for _ in range(steps):
        energies.append(float(state @ weight @ state))
        state = step_operator @ state
        if not np.isfinite(state).all(): break
    energies = np.asarray(energies)
    positive = energies[energies > 0]
    increments = np.diff(energies)
    return {
        "operator": operator.name, "family": operator.family, "energy_available": True,
        "weight_is_positive_definite": bool(np.all(np.linalg.eigvalsh(weight) > 0)),
        "dt": dt, "steps_completed": int(len(energies)),
        "initial_energy": float(energies[0]),
        "final_energy": float(energies[-1]),
        "max_energy_increase": float(np.max(increments)) if increments.size else 0.0,
        "energy_monotone_nonincreasing": bool(increments.size and np.all(increments <= 1e-12 * abs(energies[0]))),
        "energy_ratio": float(energies[-1] / energies[0]) if energies[0] else float("nan"),
        "history": energies[:: max(1, len(energies) // 200)].tolist(),
    }

In [8]:
focus = [op for op in operators if op.family in ("promoted", "reference", "followup")]
print(f"{len(focus)} operators analysed:",
      pd.Series([op.family for op in focus]).value_counts().to_dict())

bounds = pd.DataFrame([{k: v for k, v in energy_growth_bound(op).items()
                        if k != "worst_case_vector"} for op in focus])
bounds.to_csv(OUTPUT_DIR / "tables" / "energy_growth_bounds.csv", index=False)
display(bounds[["operator", "family", "weight_positive_definite", "weight_min_eigenvalue",
                "max_energy_growth_rate", "scaled_growth_rate", "energy_decay_guaranteed"]])

histories = [energy_history(op, steps=800) for op in focus]
history_table = pd.DataFrame([{k: v for k, v in h.items() if k != "history"} for h in histories])
history_table.to_csv(OUTPUT_DIR / "tables" / "energy_histories.csv", index=False)
display(history_table[["operator", "energy_monotone_nonincreasing", "max_energy_increase",
                       "energy_ratio", "dt"]])

figure, axis = plt.subplots(figsize=(7.2, 4.4))
for h in histories:
    if not h.get("energy_available"): continue
    series = np.asarray(h["history"], float)
    series = series / abs(series[0]) if series[0] else series
    style = "--" if h["family"] == "reference" else "-"
    axis.plot(np.linspace(0, 1, len(series)), series, style, linewidth=1.4,
              label=h["operator"][:40])
axis.set_xlabel("normalised time"); axis.set_ylabel("weighted energy (normalised)")
axis.set_title("Weighted energy from the worst-case initial datum")
axis.axhline(1.0, color="0.7", linewidth=0.8)
axis.legend(fontsize=6, loc="best"); figure.tight_layout()
figure.savefig(OUTPUT_DIR / "figures" / "energy_worst_case.png", dpi=200)
plt.close(figure)
print("wrote", OUTPUT_DIR / "figures" / "energy_worst_case.png")

11 operators analysed: {'promoted': 4, 'reference': 4, 'followup': 3}


,operator,family,weight_positive_definite,weight_min_eigenvalue,max_energy_growth_rate,scaled_growth_rate,energy_decay_guaranteed
0,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,True,0.001955,-0.049348,-0.000029,True
1,illumination_prog_k6_block_psd_reflection_hybr...,promoted,True,0.001679,-0.049348,-0.000040,True
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,promoted,True,0.001659,-0.049348,-0.000030,True
3,structure_only_prog_k6_block_psd_reflection_mi...,promoted,True,0.002441,-0.049348,-0.000026,True
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,True,0.004653,-0.049348,-0.000030,True
5,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,True,0.002419,-0.049348,-0.000020,True
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,True,0.003646,-0.049348,-0.000040,True
7,reference_mole_k2_m200,reference,True,0.005000,-0.049331,-0.000053,True
8,reference_mole_k4_m200,reference,True,0.003757,-0.048946,-0.000039,True
9,reference_mole_k6_m200,reference,True,0.002320,26.000485,0.018853,False


,operator,energy_monotone_nonincreasing,max_energy_increase,energy_ratio,dt
0,full_metrics_prog_k6_block_psd_reflection_hybr...,True,-4.151684e-07,0.931239,0.000005
1,illumination_prog_k6_block_psd_reflection_hybr...,True,-4.151684e-07,0.931239,0.000005
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,True,-4.151625e-07,0.931240,0.000005
3,structure_only_prog_k6_block_psd_reflection_mi...,True,-4.151684e-07,0.931239,0.000005
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,True,-4.089675e-07,0.932305,0.000004
5,v12_1_random_prog_k6_block_psd_reflection_min_...,True,-3.941661e-07,0.934845,0.000004
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,True,-4.151684e-07,0.931239,0.000005
7,reference_mole_k2_m200,True,-5.411956e-07,0.909276,0.000006
8,reference_mole_k4_m200,True,-4.255163e-07,0.929451,0.000005
9,reference_mole_k6_m200,False,1.458312e-04,0.009198,0.000005


wrote downstream_analysis/figures/energy_worst_case.png


## 4. Explicit time integration

Whether the non-real boundary modes actually restrict the stable time step is a separate
question from the energy identity, and it deserves a measured answer rather than an assumed one.
The largest stable step is found by bisection on the Runge–Kutta stability polynomial, and the
binding eigenvalue is recorded so the restriction can be attributed to the interior stencil or
to the boundary modes.

In [9]:
def time_step_table(operators: list[Operator], schemes=("rk4", "rk2")) -> list[dict[str, Any]]:
    rows = []
    for operator in operators:
        interior = operator.interior_laplacian()
        h = 1.0 / operator.cells
        eigenvalues = np.linalg.eigvals(interior)
        nonreal = np.abs(eigenvalues.imag) > 1e-8 * np.max(np.abs(eigenvalues))
        row = {"operator": operator.name, "family": operator.family, "cells": operator.cells,
               "nonreal_modes": int(nonreal.sum())}
        for scheme in schemes:
            dt, binding = max_stable_step(eigenvalues, scheme)
            row[f"dt_max_{scheme}"] = dt
            row[f"dt_max_{scheme}_over_h2"] = dt / (h * h)
            row[f"binding_mode_is_nonreal_{scheme}"] = bool(abs(binding.imag) > 1e-8 * abs(binding))
        rows.append(row)
    return rows

In [10]:
steps = pd.DataFrame(time_step_table(focus))
steps.to_csv(OUTPUT_DIR / "tables" / "time_step_limits.csv", index=False)
display(steps[["operator", "family", "nonreal_modes", "dt_max_rk4_over_h2",
               "binding_mode_is_nonreal_rk4", "dt_max_rk2_over_h2"]])
binding = bool(steps["binding_mode_is_nonreal_rk4"].any())
print("Non-real modes set the time-step limit for at least one operator:", binding)
if not binding:
    print("The stable step is governed by the interior spectral radius, so the certified "
          "operators carry no time-step advantage. Report this as a null result rather "
          "than omitting it.")

,operator,family,nonreal_modes,dt_max_rk4_over_h2,binding_mode_is_nonreal_rk4,dt_max_rk2_over_h2
0,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,0,0.451694,False,0.324340
1,illumination_prog_k6_block_psd_reflection_hybr...,promoted,0,0.451694,False,0.324340
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,promoted,0,0.451687,False,0.324335
3,structure_only_prog_k6_block_psd_reflection_mi...,promoted,0,0.451694,False,0.324340
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,0,0.444439,False,0.319132
5,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,0,0.427191,False,0.306747
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,0,0.451694,False,0.324342
7,reference_mole_k2_m200,reference,0,0.603034,False,0.433011
8,reference_mole_k4_m200,reference,0,0.463817,False,0.333048
9,reference_mole_k6_m200,reference,4,0.451694,False,0.324340


Non-real modes set the time-step limit for at least one operator: False
The stable step is governed by the interior spectral radius, so the certified operators carry no time-step advantage. Report this as a null result rather than omitting it.


## 5. Conditioning and the availability of a symmetric solve

A certified positive norm does more than license an energy estimate: it produces a genuinely
symmetric system $-Q_I L_I$ that can be handed to a conjugate-gradient solver. Without one, the
symmetric part is a different operator and solving it answers a different question, so the
practical route is a non-symmetric iteration. The symmetrisation residual measures how far each
operator is from admitting the symmetric solve at all.

In [11]:
def conjugate_gradient(matrix: np.ndarray, rhs: np.ndarray, tolerance: float = 1e-10,
                       max_iterations: int = 5000) -> tuple[np.ndarray, int, bool]:
    x = np.zeros_like(rhs); r = rhs - matrix @ x; p = r.copy()
    rs = float(r @ r); target = tolerance * math.sqrt(max(float(rhs @ rhs), 1e-300))
    for iteration in range(1, max_iterations + 1):
        Ap = matrix @ p
        denominator = float(p @ Ap)
        if denominator == 0: return x, iteration, False
        alpha = rs / denominator
        x += alpha * p; r -= alpha * Ap
        rs_new = float(r @ r)
        if math.sqrt(rs_new) <= target: return x, iteration, True
        p = r + (rs_new / rs) * p; rs = rs_new
    return x, max_iterations, False


def solver_table(operators: list[Operator], tolerance: float = 1e-10) -> list[dict[str, Any]]:
    """Condition number and CG iterations on the symmetrised Dirichlet system.

    A certified positive norm gives a genuinely symmetric system to hand to a
    conjugate-gradient solver; without one the practical route is a non-symmetric
    iteration. This is where better norm conditioning is supposed to pay for itself.
    """
    rows = []
    for operator in operators:
        interior = operator.interior_laplacian()
        weight = operator.interior_norm()
        row = {"operator": operator.name, "family": operator.family,
               "cond_L_interior": float(np.linalg.cond(interior))}
        if weight is not None:
            weight = 0.5 * (weight + weight.T)
            symmetric = -weight @ interior
            symmetric = 0.5 * (symmetric + symmetric.T)
            eigenvalues = np.linalg.eigvalsh(symmetric)
            row["symmetrised_is_spd"] = bool(np.all(eigenvalues > 0))
            row["cond_symmetrised"] = float(abs(eigenvalues[-1] / eigenvalues[0])) if eigenvalues[0] != 0 else float("inf")
            row["symmetrisation_residual"] = float(
                np.max(np.abs(weight @ interior - (weight @ interior).T)) /
                max(np.max(np.abs(weight @ interior)), 1e-300))
            # CG is only meaningful when Q L is genuinely symmetric; otherwise the
            # symmetric part is a different operator and solving it answers nothing.
            row["symmetric_system_available"] = bool(
                row["symmetrised_is_spd"] and row["symmetrisation_residual"] < 1e-10)
            if row["symmetric_system_available"]:
                grid = (np.arange(1, interior.shape[0] + 1) - 0.5) / operator.cells
                rhs = weight @ np.sin(np.pi * grid)
                _, iterations, converged = conjugate_gradient(symmetric, rhs, tolerance)
                row["cg_iterations"] = iterations
                row["cg_converged"] = converged
        rows.append(row)
    return rows

In [12]:
solver = pd.DataFrame(solver_table(focus))
solver.to_csv(OUTPUT_DIR / "tables" / "solver_conditioning.csv", index=False)
columns = [c for c in ["operator", "family", "cond_L_interior", "symmetrisation_residual",
                       "symmetrised_is_spd", "symmetric_system_available", "cond_symmetrised",
                       "cg_iterations", "cg_converged"] if c in solver]
display(solver[columns])

,operator,family,cond_L_interior,symmetrisation_residual,symmetrised_is_spd,symmetric_system_available,cond_symmetrised,cg_iterations,cg_converged
0,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,24991.318680,8.375561e-17,True,True,34299.193186,115.0,True
1,illumination_prog_k6_block_psd_reflection_hybr...,promoted,24991.348209,1.287441e-16,True,True,24991.194988,113.0,True
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,promoted,25086.980659,8.435232e-17,True,True,33767.720214,118.0,True
3,structure_only_prog_k6_block_psd_reflection_mi...,promoted,24991.350816,1.601896e-16,True,True,37795.783702,115.0,True
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,25416.957524,1.388482e-16,True,True,33159.040937,113.0,True
5,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,26456.885890,6.743658e-17,True,True,50572.459192,117.0,True
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,24991.189008,1.363824e-16,True,True,24991.162597,112.0,True
7,reference_mole_k2_m200,reference,18775.536367,8.333333e-02,True,False,18763.712007,NaN,NaN
8,reference_mole_k4_m200,reference,24383.371553,1.300632e-01,True,False,25516.438975,NaN,NaN
9,reference_mole_k6_m200,reference,26122.963975,3.566712e-01,False,False,53.040725,NaN,NaN


## 6. Summary for the manuscript

One JSON report, plus the sentences the Results and Discussion need with every number recomputed
from the stored operators. Counts are reported over distinct operators, so a duplicated archive
cannot inflate them.

In [13]:
certified_families = ("promoted", "followup")
certified = bounds[bounds.family.isin(certified_families)]
reference = bounds[bounds.family == "reference"]
growing = reference[~reference.energy_decay_guaranteed]

summary = {
    "package": str(PACKAGE_ROOT),
    "operators_analysed": int(len(focus)),
    "provenance_counts": pd.Series([op.family for op in focus]).value_counts().to_dict(),
    "novelty": novelty_gate,
    "energy": {
        "certified_with_guaranteed_decay": int(certified.energy_decay_guaranteed.sum()),
        "certified_total": int(len(certified)),
        "reference_with_guaranteed_decay": int(reference.energy_decay_guaranteed.sum()),
        "reference_total": int(len(reference)),
        "reference_operators_admitting_growth": growing.operator.tolist(),
        "worst_reference_growth_rate": float(reference.max_energy_growth_rate.max()),
    },
    "time_stepping": {
        "nonreal_modes_bind": bool(steps["binding_mode_is_nonreal_rk4"].any()),
        "dt_ratio_certified_over_reference_rk4": float(
            steps.loc[steps.family.isin(certified_families), "dt_max_rk4_over_h2"].median() /
            steps.loc[steps.family == "reference", "dt_max_rk4_over_h2"].median()),
    },
    "solver": {
        "symmetric_system_available_certified": int(
            solver.loc[solver.family.isin(certified_families), "symmetric_system_available"].sum()),
        "symmetric_system_available_reference": int(
            solver.loc[solver.family == "reference", "symmetric_system_available"].sum()),
        "median_cg_iterations_certified": float(
            solver.loc[solver.family.isin(certified_families), "cg_iterations"].median()),
    },
}
(OUTPUT_DIR / "downstream_report.json").write_text(json.dumps(summary, indent=2, default=str))

print("Sentences for the manuscript\n" + "-" * 62)
print(f"1. Under the intended weights, {len(growing)} of {len(reference)} reference closures "
      f"({', '.join(re.sub(r'_m[0-9]+$', '', g) for g in growing.operator)}) admit initial data whose "
      f"discrete energy grows, the worst at rate "
      f"{summary['energy']['worst_reference_growth_rate']:.3g}, while "
      f"{summary['energy']['certified_with_guaranteed_decay']} of "
      f"{summary['energy']['certified_total']} certified constructions guarantee decay.")
print(f"2. The stable explicit time step is set by the interior spectral radius in every case "
      f"(certified/reference median ratio "
      f"{summary['time_stepping']['dt_ratio_certified_over_reference_rk4']:.3f}); the non-real "
      f"boundary modes do not restrict it.")
print(f"3. A symmetric conjugate-gradient system is available for "
      f"{summary['solver']['symmetric_system_available_certified']} certified operators and "
      f"{summary['solver']['symmetric_system_available_reference']} reference operators, "
      f"converging in a median of "
      f"{summary['solver']['median_cg_iterations_certified']:.0f} iterations.")
print("-" * 62)
print("novelty claim supported:", novelty_gate["novelty_claim_supported"])

Sentences for the manuscript
--------------------------------------------------------------
1. Under the intended weights, 2 of 4 reference closures (reference_mole_k6, reference_mole_k8) admit initial data whose discrete energy grows, the worst at rate 845, while 7 of 7 certified constructions guarantee decay.
2. The stable explicit time step is set by the interior spectral radius in every case (certified/reference median ratio 0.987); the non-real boundary modes do not restrict it.
3. A symmetric conjugate-gradient system is available for 7 certified operators and 0 reference operators, converging in a median of 115 iterations.
--------------------------------------------------------------
novelty claim supported: False


## 7. Download the generated files

Everything written by this notebook is collected into one archive. In Colab the browser download
starts automatically; elsewhere a link or the path is printed.

In [14]:
artifacts = sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file())
for path in artifacts: print(path)

bundle_path = Path(shutil.make_archive("downstream_analysis_results", "zip",
                                       root_dir=".", base_dir=str(OUTPUT_DIR)))
print(f"\n{len(artifacts)} files, archive {bundle_path.name} "
      f"({bundle_path.stat().st_size / 1e3:.1f} kB)")

try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception:
    try:
        from IPython.display import FileLink, display as _display
        _display(FileLink(str(bundle_path)))
    except Exception:
        print("Download the archive from:", bundle_path.resolve())

downstream_analysis/downstream_report.json
downstream_analysis/figures/energy_worst_case.png
downstream_analysis/novelty_library_coefficient_template.json
downstream_analysis/novelty_library_declaration_template.json
downstream_analysis/tables/energy_growth_bounds.csv
downstream_analysis/tables/energy_histories.csv
downstream_analysis/tables/novelty_block_comparisons.csv
downstream_analysis/tables/novelty_fingerprint_distances.csv
downstream_analysis/tables/novelty_gate.json
downstream_analysis/tables/solver_conditioning.csv
downstream_analysis/tables/time_step_limits.csv

11 files, archive downstream_analysis_results.zip (136.6 kB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>